# Phase 4 - Entity Normalization

This notebook reviews the derived entity layer built from `clean_insider_transactions`.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DB_PATH = PROJECT_ROOT / 'data' / 'processed' / 'insider_research.db'
conn = sqlite3.connect(DB_PATH)

## Why CIK is the canonical key

Names and tickers can change or collide. SEC CIKs are the stable internal identifiers for insiders and issuers in this derived layer.

In [ ]:
pd.read_sql_query("""
SELECT 'insiders' AS entity, COUNT(*) AS rows FROM clean_insiders
UNION ALL SELECT 'companies', COUNT(*) FROM company_dimension
UNION ALL SELECT 'role_relationships', COUNT(*) FROM insider_company_roles
UNION ALL SELECT 'identifier_history', COUNT(*) FROM company_identifier_history
""", conn)

## Insider dimension

In [ ]:
pd.read_sql_query("""
SELECT insider_cik, canonical_name, transaction_count, company_count,
       most_common_role, most_common_role_group, distinct_raw_name_count
FROM clean_insiders
ORDER BY transaction_count DESC
LIMIT 20
""", conn)

## Insider name aliases

In [ ]:
pd.read_sql_query("""
SELECT * FROM insider_name_aliases
WHERE insider_cik IN (
  SELECT insider_cik FROM clean_insiders
  WHERE distinct_raw_name_count > 1
  ORDER BY transaction_count DESC LIMIT 10
)
ORDER BY insider_cik, is_canonical DESC, occurrence_count DESC
""", conn)

## Same-name / different-CIK cases

In [ ]:
pd.read_csv(PROJECT_ROOT / 'data' / 'exports' / 'entity_normalization' / 'insider_same_name_different_cik.csv').head(25)

## Role history

In [ ]:
pd.read_sql_query("""
SELECT insider_cik, issuer_cik, normalized_role, role_group,
       first_seen_date, last_seen_date, transaction_count
FROM insider_company_roles
ORDER BY transaction_count DESC
LIMIT 25
""", conn)

## Company dimension

In [ ]:
pd.read_sql_query("""
SELECT issuer_cik, canonical_company_name, current_or_latest_ticker,
       transaction_count, unique_insider_count, distinct_name_count, distinct_ticker_count
FROM company_dimension
ORDER BY transaction_count DESC
LIMIT 20
""", conn)

## Company name history

In [ ]:
pd.read_csv(PROJECT_ROOT / 'data' / 'exports' / 'entity_normalization' / 'company_name_variants.csv').head(25)

## Ticker history

In [ ]:
pd.read_csv(PROJECT_ROOT / 'data' / 'exports' / 'entity_normalization' / 'company_ticker_history.csv').head(25)

## Ticker reuse across companies

In [ ]:
pd.read_csv(PROJECT_ROOT / 'data' / 'exports' / 'entity_normalization' / 'tickers_multiple_ciks.csv').head(25)

## Entity linkage coverage

In [ ]:
pd.read_csv(PROJECT_ROOT / 'data' / 'exports' / 'entity_normalization' / 'entity_linkage_coverage.csv')

## QA / review candidates

In [ ]:
pd.read_csv(PROJECT_ROOT / 'data' / 'exports' / 'entity_normalization' / 'entity_review_candidates.csv').head(50)

## Readiness for market-data enrichment

The entity layer preserves historical company names and tickers. External metadata fields in `company_dimension` remain null until a later enrichment phase.